# HBV hydrological model forced with DestinE SSP370 forcing data
In this notebook we will generate forcing data for the HBV hydrological model from the output of project DestinE, simulating the SSP370 CMIP6 scenario climate model using the eWaterCycle platform. eWaterCycles integration of DestinE data is still quite new 

The code in this notebook is nearly identical to the code in [this notebook](step_1a_generate_historical_forcing.ipynb) where we generated both ERA5 and historical CMIP6 forcing data. For detailed descriptions, see that notebook.

In [1]:
# General python
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import json
from esmvalcore.config import CFG
import time
import sys

# Niceties
from rich import print

# destinE
# from forcing_destine import DestinEForcing
# from dest_auth import authenticate as dest_auth

# This will work normally for HPC
try:
    from scripts.forcing_destine import DestinEForcing
    from scripts.dest_auth import authenticate as dest_auth
except ImportError:
    # Add the project root to Python path for use on SRC
    project_root = Path().resolve().parent
    sys.path.append(str(project_root))
    
    from scripts.forcing_destine import DestinEForcing
    from scripts.dest_auth import authenticate as dest_auth

dest_auth()

ModuleNotFoundError: No module named 'geopandas'

In [ ]:
# General eWaterCycle
import ewatercycle
import ewatercycle.forcing

In [ ]:
# Load settings
# Read from the JSON file
with open('settings.json', "r") as json_file:
    settings = json.load(json_file)

In [ ]:
region_id = settings["caravan_id"]
country = settings["country"] 
seamless = settings["seamless"]

In [ ]:
display(settings)

## DestinE SSP370 future forcing

The DestinE Digital Twin for Climate Change Adaptation (Climate DT) supports adaptation activities by providing innovative climate information on multi-decadal timescales, globally, at scales at which many impacts of climate change are observed. It combines cutting-edge global Earth-system models, impact-sector applications and observations into a unified framework to provide global climate projections and impact-sector information on multi-decadal timescales (1990 to ~2050), at very high spatial resolutions (5 to 10 km). The data from the first prototype projections is already available through the DestinE platform for users with upgraded access. 

eWaterCycle is in partnership with project DestinE, and this is placeholder function. We are working with ESMValTool to facilitate working with the zarr data that DestinE provides.

In [ ]:
# Generate forcing:
try:
    DestinE_forcing_object = DestinEForcing.load(settings['path_DestinE'])
except:
    DestinE_forcing_object = DestinEForcing.generate(
       start_time=settings['future_start_date'],
       end_time=settings['future_end_date'],
       directory=settings['path_DestinE'],
       shape=settings['path_shape'],
    )

display(DestinE_forcing_object)

# Quick plot of the precipitation and potential evaporation data
plt.figure()
ds_DestinE = xr.open_mfdataset([DestinE_forcing_object['pr'],DestinE_forcing_object['evspsblpot']])
ds_DestinE["pr"].plot(label = 'precipitation')
ds_DestinE["evspsblpot"].plot(label = 'potential evaporation')
plt.legend()
plt.title('model: DestinE SSP370') 